## Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

base = r'C:\Repos\cod_esports_ml\data_clean'

maps = pd.read_csv(f'{base}/map_level.csv')
hp   = pd.read_csv(f'{base}/player_stats_hp.csv')
snd  = pd.read_csv(f'{base}/player_stats_snd.csv')
ovld = pd.read_csv(f'{base}/player_stats_ovld.csv')

print(f"Loaded — maps: {maps.shape}, hp: {hp.shape}, snd: {snd.shape}, ovld: {ovld.shape}")

## Series Labels

In [ ]:
# ── Step 1: Series Labels ────────────────────────────────────────────────────
match_teams = maps[['match_id', 'away_team', 'home_team']].drop_duplicates()

map_wins = (maps.groupby(['match_id', 'map_winner'])
                .size()
                .reset_index(name='wins'))

wins_pivot = (map_wins.pivot(index='match_id', columns='map_winner', values='wins')
                      .fillna(0))

def get_wins(match_id, team):
    return wins_pivot.loc[match_id, team] if team in wins_pivot.columns else 0

series_df = match_teams.copy()
series_df['away_wins'] = series_df.apply(lambda r: get_wins(r['match_id'], r['away_team']), axis=1)
series_df['home_wins'] = series_df.apply(lambda r: get_wins(r['match_id'], r['home_team']), axis=1)
series_df['label']     = (series_df['away_wins'] > series_df['home_wins']).astype(int)

# ── Step 2: Feature Engineering ──────────────────────────────────────────────
def agg_mode(df, mode_name, sum_cols, rate_cols):
    """Aggregate player stats → per-series team features.

    sum_cols  : summed across players per team per map, then averaged across maps.
    rate_cols : averaged across players per team per map, then averaged across maps.
    Also computes K/D ratio and non-traded-kill rate from series totals.
    """
    avail_sum  = [c for c in sum_cols  if c in df.columns]
    avail_rate = [c for c in rate_cols if c in df.columns]

    # Player rows → team totals per map
    grp   = df.groupby(['match_id', 'map_number', 'team'])
    parts = []
    if avail_sum:
        parts.append(grp[avail_sum].sum())
    if avail_rate:
        parts.append(grp[avail_rate].mean())
    team_map = pd.concat(parts, axis=1).reset_index()

    all_avail = avail_sum + avail_rate
    totals    = team_map.groupby(['match_id', 'team'])[all_avail].sum()

    feat_parts = []
    if avail_sum:
        feat_parts.append(team_map.groupby(['match_id', 'team'])[avail_sum].mean())
    if avail_rate:
        feat_parts.append(team_map.groupby(['match_id', 'team'])[avail_rate].mean())
    feat = pd.concat(feat_parts, axis=1)
    feat.columns = [f'{mode_name}_{c}' for c in feat.columns]

    # Derived ratio features (computed from series totals for accuracy)
    if 'kills' in avail_sum and 'deaths' in avail_sum:
        feat[f'{mode_name}_kd'] = totals['kills'] / totals['deaths'].replace(0, np.nan)
    if 'non_traded_kills' in avail_sum and 'kills' in avail_sum:
        feat[f'{mode_name}_nontrade_rate'] = (
            totals['non_traded_kills'] / totals['kills'].replace(0, np.nan))

    return feat.reset_index()


# Hardpoint
hp_feat = agg_mode(hp, 'hp',
    sum_cols  = ['kills', 'deaths', 'assists', 'non_traded_kills', 'damage',
                 'hill_time_sec', 'obj_kills', 'contested_hill_time_sec'],
    rate_cols = ['highest_streak', 'kills_per_hill', 'dmg_per_hill'])

# Search & Destroy
snd_feat = agg_mode(snd, 'snd',
    sum_cols  = ['kills', 'deaths', 'assists', 'non_traded_kills', 'damage',
                 'bombs_planted', 'bombs_defused', 'first_bloods', 'first_deaths'],
    rate_cols = ['highest_streak', 'kills_per_round', 'dmg_per_round'])

# SND: first blood rate = first_bloods / (first_bloods + first_deaths)
# Every round produces exactly one first blood and one first death, so this
# approximates first_bloods / total_rounds_played.
snd_fb = snd.groupby(['match_id', 'team'])[['first_bloods', 'first_deaths']].sum().reset_index()
snd_fb['snd_first_blood_rate'] = (
    snd_fb['first_bloods'] /
    (snd_fb['first_bloods'] + snd_fb['first_deaths']).replace(0, np.nan))
snd_feat = snd_feat.merge(snd_fb[['match_id', 'team', 'snd_first_blood_rate']],
                           on=['match_id', 'team'])

# Control / Overload
ovld_feat = agg_mode(ovld, 'ovld',
    sum_cols  = ['kills', 'deaths', 'assists', 'non_traded_kills', 'damage',
                 'overloads', 'obj_kills', 'score'],
    rate_cols = ['highest_streak', 'kills_per_minute', 'accuracy_pct', 'highest_multikill'])

# ── Combine into one row per series ──────────────────────────────────────────
all_feat = (hp_feat
            .merge(snd_feat,  on=['match_id', 'team'], how='outer')
            .merge(ovld_feat, on=['match_id', 'team'], how='outer'))

feat_cols = [c for c in all_feat.columns if c not in ('match_id', 'team')]
base_df   = series_df[['match_id', 'away_team', 'home_team', 'label']]

t1 = (base_df[['match_id', 'away_team']]
      .merge(all_feat, left_on=['match_id', 'away_team'], right_on=['match_id', 'team'])
      .drop(columns=['away_team', 'team'])
      .rename(columns={c: f't1_{c}' for c in feat_cols}))

t2 = (base_df[['match_id', 'home_team']]
      .merge(all_feat, left_on=['match_id', 'home_team'], right_on=['match_id', 'team'])
      .drop(columns=['home_team', 'team'])
      .rename(columns={c: f't2_{c}' for c in feat_cols}))

data = (base_df[['match_id', 'label']]
        .merge(t1, on='match_id')
        .merge(t2, on='match_id'))

# Difference features (t1 − t2) — often more predictive than raw values
for c in feat_cols:
    if f't1_{c}' in data.columns and f't2_{c}' in data.columns:
        data[f'diff_{c}'] = data[f't1_{c}'] - data[f't2_{c}']

# ── Step 3: Handle NULLs ─────────────────────────────────────────────────────
X_cols = [c for c in data.columns if c not in ('match_id', 'label')]
X = data[X_cols].copy()
y = data['label'].values

null_pct  = X.isnull().mean()
drop_cols = null_pct[null_pct > 0.5].index.tolist()
if drop_cols:
    print(f"Dropping {len(drop_cols)} columns with >50% NULLs:\n  {drop_cols}")
X = X.drop(columns=drop_cols)
X = X.fillna(X.mean())

print(f"Feature matrix: {X.shape[0]} series × {X.shape[1]} features")
print(f"NULLs remaining: {X.isnull().sum().sum()}")

# ── Step 4: XGBoost 5-Fold Cross-Validation ──────────────────────────────────
clf = XGBClassifier(
    n_estimators=100, max_depth=3, learning_rate=0.1,
    random_state=42, eval_metric='logloss', verbosity=0)

cv        = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(clf, X, y, cv=cv, scoring='accuracy')

print(f"\n{'='*52}")
print(f"  Number of series:      {len(y)}")
print(f"  Class balance:")
print(f"    Away wins (label=1): {(y==1).sum()}  ({(y==1).mean():.1%})")
print(f"    Home wins (label=0): {(y==0).sum()}  ({(y==0).mean():.1%})")
print(f"\n  CV accuracy per fold:  {cv_scores.round(3)}")
print(f"  Mean accuracy:         {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")
print(f"{'='*52}")

In [ ]:
# ── Step 5-6: Feature Importance & Confusion Matrix ─────────────────────────
clf.fit(X, y)

importances = pd.Series(clf.feature_importances_, index=X.columns)
top20 = importances.nlargest(20).sort_values()

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Top-20 feature importance bar chart
top20.plot(kind='barh', ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Top 20 Feature Importances (XGBoost)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Importance Score')
axes[0].tick_params(axis='y', labelsize=9)
axes[0].grid(axis='x', alpha=0.3)

# Confusion matrix — fitted on full dataset (in-sample; CV gives true accuracy)
y_pred = clf.predict(X)
cm = confusion_matrix(y, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1],
            xticklabels=['Home wins', 'Away wins'],
            yticklabels=['Home wins', 'Away wins'])
axes[1].set_title('Confusion Matrix (Full-Dataset Fit)', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Actual')
axes[1].set_xlabel('Predicted')

plt.tight_layout()
out_path = r'C:\Repos\cod_esports_ml\notebooks\feature_importance.png'
plt.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Plot saved → {out_path}")